# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [2]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
import json

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

print("All libraries imported successfully.")


In [ ]:
load_dotenv(find_dotenv())

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
VOCAREUM_BASE_URL = "https://openai.vocareum.com/v1"

print(f"OPENAI_API_KEY loaded: {'yes' if OPENAI_API_KEY else 'NO - check .env'}")
print(f"TAVILY_API_KEY loaded:  {'yes' if TAVILY_API_KEY else 'NO - check .env'}")


In [ ]:
# ── ChromaDB ──────────────────────────────────────────────────────────────────
# Reuse the persistent collection created in Notebook 01
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-3-small",
    api_base=VOCAREUM_BASE_URL if OPENAI_API_KEY.startswith("voc-") else None
)

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection(name="udaplay", embedding_function=embedding_fn)

print(f"Collection 'udaplay' loaded. Documents: {collection.count()}")

# ── EvaluationReport ──────────────────────────────────────────────────────────
# Pydantic model used by the evaluate_retrieval tool (LLM-as-judge structured output)
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents are sufficient to answer the question")
    description: str = Field(description="Detailed explanation of why the documents are or are not useful")

# LLM judge instance — used inside the evaluate_retrieval tool
judge_llm = LLM(model="gpt-4o-mini", temperature=0.0)
print("EvaluationReport model and judge LLM ready.")


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
@tool
def retrieve_game(query: str) -> str:
    """
    Semantic search: Finds the most relevant results in the vector DB.

    args:
    - query: a question about the game industry.

    You'll receive results as a list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(
        query_texts=[query],
        n_results=5,
        include=["documents", "metadatas", "distances"]
    )

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    formatted = []
    for doc, meta, dist in zip(documents, metadatas, distances):
        formatted.append({
            "Platform": meta.get("Platform", "Unknown"),
            "Name": meta.get("Name", "Unknown"),
            "YearOfRelease": meta.get("YearOfRelease", "Unknown"),
            "Description": meta.get("Description", ""),
            "Genre": meta.get("Genre", ""),
            "Publisher": meta.get("Publisher", ""),
            "similarity_score": round(1 - dist, 4),
        })

    return json.dumps(formatted, ensure_ascii=False, indent=2)

print(f"Tool '{retrieve_game.name}' defined.")


#### Evaluate Retrieval Tool

In [ ]:
@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> str:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    args:
    - question: original question from the user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query.\n"
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"User Question: {question}\n\n"
        f"Retrieved Documents:\n{retrieved_docs}\n\n"
        "Evaluate strictly: if the documents do not directly address the question, "
        "mark useful as false so a web search will be triggered."
    )

    response = judge_llm.invoke(
        input=prompt,
        response_format=EvaluationReport
    )

    # response_format returns a parsed object via .parsed attribute
    if hasattr(response, "parsed") and response.parsed is not None:
        report: EvaluationReport = response.parsed
    else:
        # Fallback: parse from content string
        try:
            data = json.loads(response.content)
            report = EvaluationReport(**data)
        except Exception:
            report = EvaluationReport(
                useful=False,
                description="Could not evaluate the retrieved documents. Falling back to web search."
            )

    return json.dumps({"useful": report.useful, "description": report.description}, ensure_ascii=False)

print(f"Tool '{evaluate_retrieval.name}' defined.")


#### Game Web Search Tool

In [ ]:
@tool
def game_web_search(question: str) -> str:
    """
    Performs a web search to find information about video games.

    args:
    - question: a question about the game industry.

    Returns search results including a direct answer and relevant web sources.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)

    search_result = client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False,
        max_results=5,
    )

    formatted = {
        "answer": search_result.get("answer", ""),
        "sources": [
            {
                "title": r.get("title", ""),
                "url": r.get("url", ""),
                "content": r.get("content", "")[:400],  # truncate for context
            }
            for r in search_result.get("results", [])[:3]
        ],
    }

    return json.dumps(formatted, ensure_ascii=False, indent=2)

print(f"Tool '{game_web_search.name}' defined.")


### Agent

In [ ]:
UDAPLAY_INSTRUCTIONS = """
You are UdaPlay, an expert AI Research Agent specialized in video games.

## WORKFLOW — Always follow this order:
1. Call `retrieve_game` with the user's question to search the internal database.
2. Call `evaluate_retrieval` with the original question AND the retrieved documents to assess quality.
3. If evaluate_retrieval returns useful=false, call `game_web_search` with the original question.
4. Synthesize a clear, well-structured answer based on all available information.

## OUTPUT FORMAT:
- Always state whether the information came from the "Internal Database" or "Web Search".
- Be concise and factual. Format your answer clearly with game title, platform, year, and any other relevant details.
- If information was found in both sources, combine them coherently.
- If the information cannot be found, say so clearly.

## IMPORTANT:
- Never skip the retrieve_game step — always check the internal database first.
- Never skip the evaluate_retrieval step — always assess the quality of what you retrieved.
- Only call game_web_search when evaluate_retrieval explicitly returns useful=false.
"""

udaplay_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=UDAPLAY_INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.2,
)

print("UdaPlay agent created successfully.")
print(f"Model: gpt-4o-mini | Tools: {[t.name for t in udaplay_agent.tools]}")


In [ ]:
def print_agent_report(query: str, run_object):
    """Extract and display a structured report from an agent run."""
    final_state = run_object.get_final_state()
    messages = final_state.get("messages", [])

    # Extract tool calls from AI messages
    tools_used = []
    for msg in messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                tools_used.append(tc.function.name)

    # Extract final answer (last AI message with content)
    final_answer = ""
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.content:
            final_answer = msg.content
            break

    # Determine source
    source = "Internal Database"
    if "game_web_search" in tools_used:
        source = "Web Search (Tavily)" if "retrieve_game" not in tools_used else "Internal Database + Web Search (Tavily)"

    print("=" * 70)
    print(f"QUERY:   {query}")
    print(f"TOOLS:   {' → '.join(tools_used) if tools_used else 'None'}")
    print(f"SOURCE:  {source}")
    print(f"TOKENS:  {final_state.get('total_tokens', 'N/A')}")
    print("-" * 70)
    print(f"ANSWER:\n{final_answer}")
    print("=" * 70)
    print()


# ── Run agent on the three example queries ────────────────────────────────────
queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for query in queries:
    run = udaplay_agent.invoke(query)
    print_agent_report(query, run)


### (Optional) Advanced

In [ ]:
# ── (Optional) Long-term memory: persist web search results ──────────────────
# This allows UdaPlay to "remember" facts discovered via web search across sessions
from lib.memory import LongTermMemory, MemoryFragment
from lib.vector_db import VectorStoreManager

# Set up the long-term memory backed by an in-memory ChromaDB store
db_manager = VectorStoreManager(openai_api_key=OPENAI_API_KEY)
long_term_memory = LongTermMemory(db=db_manager)

def save_web_findings_to_memory(query: str, answer: str, owner: str = "udaplay"):
    """Save a web-search result to long-term memory for future retrieval."""
    content = f"Q: {query}\nA: {answer}"
    fragment = MemoryFragment(content=content, owner=owner, namespace="game_facts")
    long_term_memory.register(fragment)
    print(f"Saved to long-term memory: '{query[:60]}...'")

# ── Demo: run one query and save the web result to memory ─────────────────────
demo_query = "Who developed The Legend of Zelda: Breath of the Wild?"
print(f"\n[Demo] Running: '{demo_query}'")

run = udaplay_agent.invoke(demo_query, session_id="ltm_demo")
final_state = run.get_final_state()

# Extract tools used and final answer
messages = final_state.get("messages", [])
tools_used = []
for msg in messages:
    if isinstance(msg, AIMessage) and msg.tool_calls:
        for tc in msg.tool_calls:
            tools_used.append(tc.function.name)

final_answer = ""
for msg in reversed(messages):
    if isinstance(msg, AIMessage) and msg.content:
        final_answer = msg.content
        break

print_agent_report(demo_query, run)

# Save to long-term memory if web search was used
if "game_web_search" in tools_used:
    save_web_findings_to_memory(demo_query, final_answer)

# Search long-term memory to demonstrate retrieval
print("\n[Long-term Memory] Searching for stored game facts...")
search_results = long_term_memory.search(
    query_text="Zelda developer",
    owner="udaplay",
    namespace="game_facts",
    limit=3
)
for fragment in search_results.fragments:
    print(f"  → {fragment.content[:120]}")
